# MultiNodeDAQ offline SCF analysis
Open a completed recording, preserve gaps, and compute the paper-based FFT accumulation SCF. Select a contiguous window with constant metadata; no missing samples are filled.

In [ ]:
from pathlib import Path
import numpy as np
from multinodedaq import open_session, export_hdf5
from multinodedaq.spectral import FAM, FAMSettings
recording = Path("../.artifacts/stage3/example/recording")  # choose your recording
session = open_session(recording)
list(session.streams)

In [ ]:
unit, acquisition_session = next(iter(session.streams))
fam = FAM(FAMSettings(rate=25000, df=10, dalpha=5))
batch = session.read_samples(unit, acquisition_session, first=0, count=fam.count)
assert not batch.gaps and not batch.flags.any()
assert all(len(np.unique(field)) == 1 for field in (batch.config, batch.calibration, batch.timing))
result = fam.compute(batch.codes, first_sample=batch.first)
result["scf"].shape, result["df_hz"], result["dalpha_hz"]

In [ ]:
# Preserve the full complex SCF, valid-grid mask, and coordinates for later plotting.
np.savez_compressed("scf.npz", frequency_hz=result["frequency_hz"], alpha_hz=result["alpha_hz"], scf=result["scf"], valid_grid=result["valid_grid"])
export_hdf5(session, "recording.h5")  # exclusive creation
session.close()